# Knowledge Agent: Hierarchical Routing Tree (SQL Base)

Dieses Jupyter Notebook initialisiert und konfiguriert das relationale Datenfundament für den **Knowledge Agent**. 

### Zweck der Architektur:
* **Hierarchisches Routing (State-Graph / Decision Tree):** Der Agent nutzt diese SQLite-Datenbank wie einen Entscheidungsbaum (ähnlich einem Random Forest), um eingehende Anfragen schrittweise zu analysieren und über Verzweigungen (Knoten) den exakten Ausführungspfad zu bestimmen.
* **Selbstdokumentation:** Die Struktur ist so ausgelegt, dass das LLM beim Start seine eigenen Navigationsregeln und Zustände ausliest.

## Architektur-Übersicht der Datenbank

Die Datenbank gliedert sich in drei zentrale Komponenten:

1. **`agent_system_manifest`**: Enthält die grundlegenden Systemanweisungen und Direktiven. Das LLM liest diese Tabelle zuerst aus, um zu verstehen, wie es den Routing-Baum zu navigieren hat.
2. **`agent_routing_tree`**: Der eigentliche Entscheidungsbaum (`node_id`, `parent_node_id`, `routing_condition`). Hier sind die logischen Verknüpfungen hinterlegt, welcher Schritt (A -> B -> C) als Nächstes ausgeführt wird.
3. **`agent_execution_memory`**: Dient als Feedback-Loop und Erfahrungsspeicher (Self-Reflection), um erfolgreiche Pfade zu gewichten und den Baum dynamisch zu erweitern.

In [1]:
# Import
import sqlite3
import os

In [2]:
# Globale Definitionen für die Ordnerstruktur
ANKER_DIR = "Offline_AI"
BASE_DIR = "Knowledge"
AGENT_SUBDIR = "knowledge_agent_hierarchical_routing_tree_sql"
# globalisierte Variablen der SQLite-Datenbankdatei
DB_FILENAME = "knowledge_agent_routing_tree.db"

In [3]:
# def für die Initialisierung der Ordnerstruktur
def initialize_find_folder():
    """
    Diese Funktion prüft, ob die notwendige Ordnerstruktur für den Knowledge Agent im Projekt vorhanden ist.
    Gibt bei jedem Teilschritt ein klares Status-Print in der Konsole aus.
    """
    print("--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---")

    # Schritt 0: Projekt-Root 'Offline_AI' ermitteln, um nicht im Notebooks-Ordner zu landen
    current_path = os.path.abspath(os.getcwd())
    print(f"--> [INFO] Start-Pfad des Notebooks: {current_path}")
    
    if ANKER_DIR.lower() in current_path.lower():
        base_parts = current_path.split(os.sep)
        anker_index = [p.lower() for p in base_parts].index(ANKER_DIR.lower())
        base_root = os.sep.join(base_parts[:anker_index + 1])
    else:
        base_root = current_path

    print(f"--> [ERFOLG] Projekt-Root '{ANKER_DIR}' identifiziert unter: {base_root}")

    # Schritt 1: Hauptverzeichnis "Knowledge" im Anker-Ordner suchen oder erstellen
    target_knowledge_dir = os.path.join(base_root, BASE_DIR)
    print(f"Suche nach Hauptverzeichnis: '{target_knowledge_dir}'...")
    if not os.path.exists(target_knowledge_dir):
        os.makedirs(target_knowledge_dir)
        print(f"--> [ERFOLG] Hauptverzeichnis '{target_knowledge_dir}' wurde neu erstellt.")
    else:
        print(f"--> [INFO] Hauptverzeichnis '{target_knowledge_dir}' wurde gefunden.")

    # Schritt 2: Unterordner für den Agenten im Knowledge-Verzeichnis suchen oder erstellen
    target_directory = os.path.join(target_knowledge_dir, AGENT_SUBDIR)
    print(f"Suche nach Unterordner: '{target_directory}'...")
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
        print(f"--> [ERFOLG] Unterordner '{AGENT_SUBDIR}' wurde neu erstellt.")
    else:
        print(f"--> [INFO] Unterordner '{AGENT_SUBDIR}' existiert bereits.")

    # Schritt 3: Vollständigen Pfad zur Datenbank zusammenbauen
    db_path = os.path.join(target_directory, DB_FILENAME)
    print(f"Pfad-Zusammenführung abgeschlossen. Zieldatei: '{db_path}'")
    print("--- [ENDE] Ordnerstruktur erfolgreich geprüft ---")
    
    return db_path

## Globale Schema-Definitionen & Tabellen-Konstanten

Bevor wir die Datenbanktabellen erstellen, definieren wir alle Tabellennamen und Spalten als globale Konstanten. Dadurch verhindern wir Tippfehler und machen den Code absolut übersichtlich.

* **Tabelle 1 (`agent_system_manifest`)**: Das Start-Manifest für das LLM (liest es zuerst aus, um seine Regeln zu verstehen).
  * Spalte 1: `directive_key` (Primärschlüssel, z. B. `core_logic`)
  * Spalte 2: `explanation_for_agent` (Die Anweisung/Erklärung für das Modell)

In [4]:
# GLOBALE VARIABLEN & SCHEMA-DEFINITIONEN: TABELLE 1
# Name der ersten Tabelle für das System-Manifest (Bedienungsanleitung für das LLM)
TABLE_MANIFEST = "agent_system_manifest"

# Globale Spaltennamen für diese Tabelle
COL_MANIFEST_KEY = "directive_key"          # Der eindeutige Schlüssel (Primary Key, z.B. 'core_logic')
COL_MANIFEST_VAL = "explanation_for_agent"    # Die eigentliche Anweisung / Erklärung für das LLM

# agent_system_manifest

In [5]:
# Erstellt agent_system_manifest Tabelle in der SQLite-Datenbank
def create_system_manifest_table(db_path):
    """
    Erstellt die erste Tabelle ('agent_system_manifest') in der SQLite-Datenbank.
    Diese Tabelle dient dem LLM als reiner Aufklärungs- und Regel-Layer, bevor es mit der Suche beginnt.
    """
    print(f"--- [START] Erstelle Tabelle '{TABLE_MANIFEST}' ---")
    
    # 1. Verbindung zur SQLite-Datenbank herstellen (unter Verwendung des übergebenen Pfads)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"--> [INFO] Verbindung zur Datenbank geöffnet: {db_path}")

    # 2. Tabelle erstellen unter Nutzung der globalen Konstanten
    create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {TABLE_MANIFEST} (
            {COL_MANIFEST_KEY} TEXT PRIMARY KEY,
            {COL_MANIFEST_VAL} TEXT NOT NULL
        )
    """
    cursor.execute(create_table_query)
    print(f"--> [ERFOLG] Tabelle '{TABLE_MANIFEST}' wurde erfolgreich angelegt (oder war bereits vorhanden).")

    # 3. Metadaten-Direktiven zur reinen Aufklärung und Regeleinhaltung für das LLM
    initial_directives = [
        ("system_purpose", "This manifest defines the mandatory rules, security constraints, and operational boundaries for the LLM before any search begins."),
        ("immutable_base_rule", "CORE PROTECTION: Base tables without timestamps are strictly immutable. Deleting or overwriting core base tables is forbidden. Appends only."),
        ("versioning_and_registry", "ADMIN CONTROL: Always check the 'meta_admin_bootstrap_registry' where is_active = 1 to resolve active table versions and prevent table collisions."),
        ("navigation_handoff", "MANIFEST COMPLETE: After understanding these rules, transition to the indexed routing tree to evaluate user intent and select the correct execution path.")
    ]

    # 4. Daten sicher in die Tabelle schreiben (INSERT OR REPLACE verhindert Duplikate)
    insert_query = f"""
        INSERT OR REPLACE INTO {TABLE_MANIFEST} ({COL_MANIFEST_KEY}, {COL_MANIFEST_VAL}) 
        VALUES (?, ?)
    """
    cursor.executemany(insert_query, initial_directives)
    print("--> [ERFOLG] Aufklärungs- und Regel-Direktiven wurden erfolgreich in das Manifest eingefügt.")

    # 5. Transaktion speichern und Verbindung schließen
    conn.commit()
    conn.close()
    print(f"--- [ENDE] Tabelle '{TABLE_MANIFEST}' erfolgreich initialisiert ---")

## Globale Schema-Definitionen & Tabellen-Konstanten: Tabelle 2 (Routing-Baum)

Tabelle 2 (`agent_routing_tree_indexed`) bildet den eigentlichen Entscheidungsbaum ab. 
* `node_id`: Die eindeutige ID des Knotens
* `parent_node_id`: Der Verweis auf den vorherigen Knoten (um den Pfad zurückzuverfolgen).
* `routing_condition`: Die Bedingung oder Absicht, wann dieser Pfad gewählt wird.
* `action_payload`: Die auszuführende Aktion oder das Skript.
* `target_table`: Eventuelle Detail-Tabellen für fachspezifische Daten.
* `success_weight`: Gewichtung/Erfahrungswert des Pfades.

# agent_routing_tree_indexed

In [6]:
# Globale Variablen & Schema-Definitionen: TABELLE 2
TABLE_ROUTING = "agent_routing_tree_indexed"

ROUTING_COLUMNS = {
    "node_id": "TEXT PRIMARY KEY",       # Lokale ID innerhalb DIESER Weichentabelle (z.B. "000")
    "parent_node_id": "TEXT",             # Übergeordneter Knoten zur Nachvollziehbarkeit
    "node_purpose": "TEXT NOT NULL",      # Beschreibung, was dieser Ast bezweckt
    "routing_condition": "TEXT NOT NULL", # Bedingung für das LLM, wann dieser Ast gewählt wird
    "action_payload": "TEXT NOT NULL",    # Nutzdaten oder Instruktionen für diesen Schritt
    "target_table": "TEXT",               # NÄCHSTE TABELLE (Der Sprung in die Zieltabelle / Ausführungstabelle)
    "target_node_id": "TEXT",             # Start-ID in der nächsten Tabelle (z.B. "000")
    "success_weight": "REAL DEFAULT 1.0",
    "execution_count": "INTEGER DEFAULT 0",
    "is_active": "INTEGER DEFAULT 1",
    "year": "INTEGER",
    "month": "INTEGER",
    "day": "INTEGER",
    "hour": "INTEGER",
    "minute": "INTEGER",
    "second": "INTEGER"
}

In [7]:
# Erstellt agent_routing_tree Tabelle in der SQLite-Datenbank
def create_routing_tree_table(db_path):
    print(f"--- [START] Prüfe und initialisiere leere Tabelle '{TABLE_ROUTING}' ---")
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print(f"--> [INFO] Verbindung zur Datenbank geöffnet: {db_path}")

    # Schritt 1: Prüfen, ob die Tabelle überhaupt existiert
    cursor.execute(f"""
        SELECT name FROM sqlite_master WHERE type='table' AND name='{TABLE_ROUTING}';
    """)
    table_exists = cursor.fetchone()

    if not table_exists:
        print(f"--> [INFO] Tabelle '{TABLE_ROUTING}' existiert noch nicht. Wird komplett neu erstellt...")
        columns_definition = ", ".join([f"{col_name} {col_type}" for col_name, col_type in ROUTING_COLUMNS.items()])
        create_table_query = f"""
            CREATE TABLE IF NOT EXISTS {TABLE_ROUTING} (
                {columns_definition}
            )
        """
        cursor.execute(create_table_query)
        print(f"--> [ERFOLG] Tabelle '{TABLE_ROUTING}' wurde erfolgreich und leer neu erstellt.")
    else:
        print(f"--> [INFO] Tabelle '{TABLE_ROUTING}' existiert bereits. Prüfe auf fehlende Spalten...")
        
        # Bestehende Spalten in der Datenbank auslesen
        cursor.execute(f"PRAGMA table_info({TABLE_ROUTING});")
        existing_columns_info = cursor.fetchall()
        existing_column_names = [col[1] for col in existing_columns_info]

        # Abgleich: Welche Spalten aus unserem Schema fehlen in der Datenbank?
        for col_name, col_type in ROUTING_COLUMNS.items():
            if col_name not in existing_column_names:
                print(f"--> [INFO] Spalte '{col_name}' fehlt in der Tabelle. Füge sie hinzu...")
                alter_query = f"ALTER TABLE {TABLE_ROUTING} ADD COLUMN {col_name} {col_type};"
                try:
                    cursor.execute(alter_query)
                    print(f"--> [ERFOLG] Spalte '{col_name}' erfolgreich hinzugefügt.")
                except Exception as e:
                    print(f"--> [FEHLER] Konnte Spalte '{col_name}' nicht hinzufügen: {e}")
            else:
                print(f"--> [OK] Spalte '{col_name}' ist bereits vorhanden.")

    conn.commit()
    conn.close()
    print(f"--- [ENDE] Leere Tabellen- und Spaltenprüfung für '{TABLE_ROUTING}' abgeschlossen ---")

## Initialisierung der SQLite-Datenbank

Im folgenden Code-Block wird die Verbindung zur lokalen SQLite-Datenbank hergestellt und die oben beschriebene Tabellenstruktur fehlerfrei aufgebaut. Falls die Datenbank noch nicht existiert, wird sie automatisch generiert.

# agent_system_manifest

In [8]:
# def verbinden die Schritte der Initialisierung und Tabellen-Erstellung
# Schritt 1: Ordnerstruktur prüfen und den exakten Pfad zur Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: System-Manifest Tabelle erstellen und füllen
create_system_manifest_table(db_path)

# Schritt 3: Die leere Routing-Tabelle inklusive Spaltenprüfung erstellen
create_routing_tree_table(db_path)

print(f"\n[ERFOLG] Alle Grundstrukturen stehen bereit. Die Datenbank ist im Ordner hinterlegt: {db_path}")

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [ERFOLG] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' wurde neu erstellt.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Erstelle Tabelle 'agent_system_manifest' ---
--> [INFO] Verbindung zur Datenban

# 🚀 Workflow-Anleitung: 
Interaktive Dateneingabe für den Routing-Tree

- Folge diesen Schritten im Jupyter Notebook, um neue Einträge sicher und strukturiert in deine Tabellen einzupflegen:

- Schritt 1: Spalten-Template generieren
Führe die Funktion aus und übergib den Namen der gewünschten Tabelle, um alle vorhandenen Spalten als fertiges Python-Dictionary auszugeben.

Aktion: Starte den Generator-Code mit deinem Tabellennamen.

Ergebnis: Du erhältst eine formatierte Textausgabe im Notebook-Output.

- Schritt 2: Template kopieren & ausfüllen
Kopiere den im Output generierten Python-Block, füge ihn in das nächste Codefeld ein und trage deine gewünschten Daten ein.

- Schritt 3: Sicherer Schreibvorgang (Insert)
Führe im nächsten Codefeld die geschützte Insert-Funktion aus und überdies dein ausgefülltes Dictionary.

    - Schutzmechanismus: Falls eine ID (z. B. "000") bereits in der Tabelle existiert, bricht der Vorgang automatisch ab und gibt eine Warnung aus.

    - Überschreiben erlauben: Falls du einen bestehenden Eintrag bewusst aktualisieren möchtest, musst du den Schutzparameter aktivieren (z. B. overwrite=True).

# agent_routing_tree_indexed

In [9]:
# Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "agent_routing_tree_indexed"


In [10]:
# Analyse der Tabellenstruktur und Ausgabe eines Python-Dictionaries mit Standardwerten
def analyze_table_structure(table_name, save_rite_status=True):
    """
    Analysiert eine SQLite-Tabelle, prüft deren Existenz und gibt ein passendes 
    Python-Dictionary (NAME_COLUMNS) mit Standardwerten und Format-Hinweisen aus.
    Fällt bei nicht existierenden Tabellen auf das globale ROUTING_COLUMNS-Schema zurück.
    """
    #print(f"--- [START] Analysiere Tabelle: '{table_name}' ---")
    
    # Exakten Pfad zur SQLite-Datenbank über die globale Ordnerstruktur-Funktion ermitteln
    db_path = initialize_find_folder()
    #print(f"--> [INFO] Ziel-Datenbank: {db_path}")
    #print(f"--> [INFO] Schreibschutz-Status (SAVE_RITE): {save_rite_status}")

    # Verbindung zur Datenbank herstellen
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    name_columns_result = {}

    try:
        # Spalteninformationen der global definierten Tabelle auslesen
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns_info = cursor.fetchall()
        
        # Fall 1: Tabelle existiert in der DB und hat Spalten
        if columns_info:
            #print(f"--> [ERFOLG] {len(columns_info)} Spalten direkt aus der DB-Tabelle '{table_name}' ausgelesen.")
            #print(" ")
            #print(" KOPIERE DIESEN BLOCK IN DAS NÄCHSTE CODEFELD UND FÜLLE IHN AUS:")
            #print(" ")
            print("NAME_ROWS= {")
            for col in columns_info:
                col_name = col[1]    # Spaltenname
                col_type = col[2]    # Datentyp (TEXT, INTEGER, etc.)
                
                # Format-Hinweise ableiten
                col_upper = col_type.upper()
                if "INT" in col_upper:
                    format_hint = "Format: Integer (Ganzzahl, z.B. 0, 1)"
                    default_val = 0
                elif "REAL" in col_upper:
                    format_hint = "Format: Float / Decimal (Dezimalzahl, z.B. 1.0)"
                    default_val = 0.0
                else:
                    format_hint = "Format: Text / String (z.B.Textbeschreibungen)"
                    default_val = ""
                    
                if col_name in ["year", "month", "day", "hour", "minute", "second"]:
                    format_hint = "Format: System-Auto-Fill (Wird automatisch vom System eingetragen, wenn leer)"

                print(f"    \"{col_name}\": {repr(default_val)},  # {format_hint}")
                name_columns_result[col_name] = default_val
                
            print("}")
            
        # Fall 2: Tabelle existiert noch nicht oder ist leer -> Nutzung des globalen Schemas (ROUTING_COLUMNS)
        else:
            #print(f"--> [HINWEIS] Die Tabelle '{table_name}' existiert noch nicht physisch in der DB.")
            #print("--> [INFO] Generiere Vorlage stattdessen aus dem globalen Python-Schema (ROUTING_COLUMNS)...")
            #print(" KOPIERE DIESEN BLOCK IN DAS NÄCHSTE CODEFELD UND FÜLLE IHN AUS:")
            #print(" ")
            print("NAME_ROWS= {")
            for col_name, col_definition in ROUTING_COLUMNS.items():
                col_upper = col_definition.upper()
                if "INT" in col_upper:
                    format_hint = "Format: Integer (Ganzzahl, z.B. 0, 1)"
                    default_val = 0
                elif "REAL" in col_upper:
                    format_hint = "Format: Float / Decimal (Dezimalzahl, z.B. 1.0)"
                    default_val = 0.0
                else:
                    format_hint = "Format: Text / String (z.B. Textbeschreibungen)"
                    default_val = ""
                    
                if col_name in ["year", "month", "day", "hour", "minute", "second"]:
                    format_hint = "Format: System-Auto-Fill (Wird automatisch vom System eingetragen, wenn leer)"

                print(f"    \"{col_name}\": {repr(default_val)},  # {format_hint}")
                name_columns_result[col_name] = default_val
                
            print("}")

    except Exception as e:
        print(f"--> [FEHLER] Konnte Spalten nicht auslesen: {e}")
    finally:
        conn.close()
        print(f"--- [ENDE] Analyse abgeschlossen ---")
        
    return #name_columns_result

In [11]:
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "parent_

In [12]:
NAME_ROWS = [
    {
        "node_id": "000",
        "parent_node_id": "ROOT",
        "node_purpose": "PROTECTED ADMIN INDEX: Points to the 'meta_admin_bootstrap_registry'. ACCESS DENIED if 'SAVE_RITE_ADMIN' is active! Defines strict rules for indexing new data to expand the library.",
        "routing_condition": "CHECK_SAVE_RITE_AND_ADMIN_PASSCODE",
        "action_payload": "DENY_ACCESS_IF_SAVE_WRITE_TRUE__ELSE_LOAD_META_ADMIN_REGISTRY",
        "target_table": "meta_admin_bootstrap_registry",
        "target_node_id": "meta_admin_bootstrap_registry_root",
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
    },
    {
        "node_id": "ZZZ",
        "parent_node_id": "999",
        "node_purpose": "GESCHÜTZTER NOTFALL-INDEX: Verweist auf die 'meta_admin_repair_registry'. Greift ein, wenn Kettenbrüche oder Fehler im Langzeitgedächtnis auftreten, und aktiviert das FX_CHAIN-Protokoll.",
        "routing_condition": "CHECK_CHAIN_INTEGRITY_AND_TRIGGER_FX",
        "action_payload": "ISOLATE_BROKEN_NODE__LOAD_REPAIR_REGISTRY__INIT_FX_CHAIN",
        "target_table": "meta_admin_repair_registry",
        "target_node_id": "000",
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0,
        "month": 0,
        "day": 0,
        "hour": 0,
        "minute": 0,
        "second": 0,
    }
]

In [13]:
# Fügt die Tabellen-Rows (z.B. INITIAL_TABOO_RULES_ROWS) dynamisch in eine angegebene Tabelle ein
import datetime

def insert_bootstrap_routing_rows(target_table_name, rows_data):
    """
    Fügt eine Liste von Routing- oder Regel-Knoten (Rows) dynamisch in eine angegebene Tabelle ein.
    Ergänzt automatisch fehlende Zeitstempel und verarbeitet mehrere Zeilen in einem Loop.
    Prüft vorab SAVE_RITE: Falls True und ein Knoten existiert, wird das Überschreiben blockiert.
    """
    print(f"--- [START] Injiziere Zeilen-Batch in Tabelle: '{target_table_name}' ---")
    
    # Exakten Pfad über die globale Ordnerstruktur-Funktion ermitteln
    db_path = initialize_find_folder()
    print(f"--> [INFO] Ziel-Datenbank: {db_path}")

    # Verbindung zur SQLite-Datenbank herstellen
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Falls nur ein einzelnes Dictionary übergeben wurde, in eine Liste packen
        if isinstance(rows_data, dict):
            rows_data = [rows_data]

        for node_data in rows_data:
            node_id = node_data.get("node_id")
            print(f"--> [VERARBEITE] Bearbeite Knoten ID: '{node_id}'")

            # Aktuelle Zeit für das Auto-Fill ermitteln
            now = datetime.datetime.now()
            time_fields = {
                "year": now.year,
                "month": now.month,
                "day": now.day,
                "hour": now.hour,
                "minute": now.minute,
                "second": now.second
            }
            
            # Zeitstempel ergänzen, falls im Dictionary 0 oder leer
            for field, val in time_fields.items():
                if field in node_data and (node_data[field] == 0 or not node_data[field]):
                    node_data[field] = val

            # Prüfen, ob der Knoten bereits in der Tabelle existiert
            cursor.execute(f"SELECT 1 FROM {target_table_name} WHERE node_id = ?", (node_id,))
            exists = cursor.fetchone()

            # SAVE_RITE Logik: Wenn der Knoten existiert und SAVE_RITE True ist -> blockieren
            save_rite_active = globals().get("SAVE_RITE", True)
            
            if exists and save_rite_active:
                print(f"----> [SECURITY ALERT] SAVE_RITE ist aktiv (True).")
                print(f"----> [BLOCKIERT] Knoten '{node_id}' existiert bereits in '{target_table_name}' und darf nicht überschrieben werden!")
            else:
                # Dynamisches SQL-Statement generieren
                columns = list(node_data.keys())
                placeholders = ["?" for _ in columns]
                values = list(node_data.values())

                insert_query = f"""
                    INSERT OR REPLACE INTO {target_table_name} ({", ".join(columns)})
                    VALUES ({", ".join(placeholders)})
                """

                cursor.execute(insert_query, values)
                print(f"----> [ERFOLG] Knoten '{node_id}' erfolgreich in '{target_table_name}' gespeichert.")

        # Alles gesammelt committen
        conn.commit()
        print(f"--- [ERFOLG] Alle Zeilen des Batches erfolgreich in '{target_table_name}' injiziert ---")

    except Exception as e:
        print(f"--> [FEHLER] Konnte Zeilen-Batch nicht in '{target_table_name}' speichern: {e}")
        conn.rollback()
    finally:
        conn.close()
        print(f"--- [ENDE] Injektion abgeschlossen ---")

In [14]:
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Injiziere Zeilen-Batch in Tabelle: 'agent_routing_tree_indexed' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] Zie

# meta_admin_bootstrap_registry

In [15]:
# Global variables & schema definitions: TABLE 2 (Meta Admin Bootstrap Registry)
TABLE_ROUTING = "meta_admin_bootstrap_registry"

ROUTING_COLUMNS = {
    "node_id": "TEXT",                      # Stays constant (e.g., "000"), identifies the logical branch (no PK to allow multiple versions)
    "parent_node_id": "TEXT",               # Parent node for traceability and lineage
    "node_purpose": "TEXT NOT NULL",        # Detailed purpose description of this admin branch
    "routing_condition": "TEXT NOT NULL",   # Condition for the LLM: Checks admin rights and write protection (SAVE_RITE)
    "action_payload": "TEXT NOT NULL",      # Instructions and payload for accessing this administrative level
    "target_table": "TEXT",                 # Target table name (protected base table or timestamp-suffixed version)
    "target_node_id": "TEXT",               # Start ID within the target table (e.g., "root_v1")
    "version": "INTEGER DEFAULT 1",         # Explicit version number (v1, v2, v3...) for safe tracking
    "success_weight": "REAL DEFAULT 1.0",   # Reliability / success weight factor of this entry
    "execution_count": "INTEGER DEFAULT 0", # Execution counter tracking how often this node was invoked
    "is_active": "INTEGER DEFAULT 1",       # Exclusive flag: Only ONE row per node_id is allowed to be active (1 = active, 0 = inactive)
    "year": "INTEGER",                      # System Auto-Fill: Creation or modification year
    "month": "INTEGER",                     # System Auto-Fill: Month
    "day": "INTEGER",                       # System Auto-Fill: Day
    "hour": "INTEGER",                      # System Auto-Fill: Hour
    "minute": "INTEGER",                    # System Auto-Fill: Minute
    "second": "INTEGER"                     # System Auto-Fill: Second
}

In [16]:
# [BOOTSTRAP-ABGLEICH] Verbindung sichern und Meta-Registry aktualisieren/einbinden
print("--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---")

# Schritt 1: Ordnerstruktur und exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, erweiterte Routing-Tabelle (Meta Admin Bootstrap Registry) prüfen, 
# bestehende Tabellen schützen und fehlende Spalten (inkl. Versionierung & Aktiv-Flags) ergänzen
create_routing_tree_table(db_path)

print(f"\n[ERFOLG] Die Meta-Registry ist aktiv, das globale Schema ist aktualisiert und die Datenbank ist bereit: {db_path}")

--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und init

In [17]:
# Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "meta_admin_bootstrap_registry"


In [18]:
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "parent_

In [19]:
# Global variables & schema definitions: TABLE 2 (Meta Admin Bootstrap Registry)

NAME_ROWS = {
    "node_id": "000",
    "parent_node_id": "root",
    "node_purpose": "MASTER_GATEWAY_TO_TABOO_RULES: Redirects the agent to the immutable governance and taboo rulebook before any admin or schema operation.",
    "routing_condition": "Must execute unconditionally as the very first step of any administrative workflow.",
    "action_payload": "MANDATORY REDIRECTION: Load the complete rulebook from meta_admin_taboo_rules (from node '000' to '999'). Absorb all prohibitions, non-destructive versioning mandates, and ensure the loop returns cleanly before altering any database structures.",
    "target_table": "meta_admin_taboo_rules",
    "target_node_id": "000",
    "version": 1,
    "success_weight": 1.0,
    "execution_count": 0,
    "is_active": 1,
    "year": 0,
    "month": 0,
    "day": 0,
    "hour": 0,
    "minute": 0,
    "second": 0
}

In [20]:
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Injiziere Zeilen-Batch in Tabelle: 'meta_admin_bootstrap_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] 

# meta_admin_taboo_rules

In [21]:
# Global variables & schema definitions: TABLE 2 (Meta Admin Taboo Rules / Long-Term Memory)
TABLE_ROUTING = "meta_admin_taboo_rules"

ROUTING_COLUMNS = {
    "node_id": "TEXT NOT NULL",                  # Stays constant (e.g., "000" to "998"), identifies the logical sequence step
    "parent_node_id": "TEXT",                    # Parent node for traceability and lineage of the build plan
    "directive_name": "TEXT NOT NULL",           # Technical name of the rule / anti-shrink directive (e.g., PRESERVE_CORE_CODE)
    "mandatory_payload": "TEXT NOT NULL",        # The exact code fragment or text snippet the agent must carry and implement
    "fallback_action": "TEXT DEFAULT 'FX_CHAIN'",  # Ultra-compact token alias: Triggers chain repair & gap healing if a sequence break occurs
    "target_table": "TEXT",                      # Target table name for the final jump (e.g., meta_admin_bootstrap_registry)
    "target_node_id": "TEXT",                    # Start ID within the target table after loop termination (e.g., "001")
    "version": "INTEGER DEFAULT 1",              # Explicit version number (v1, v2, v3...) for safe tracking
    "success_weight": "REAL DEFAULT 1.0",        # Reliability / success weight factor of this entry
    "execution_count": "INTEGER DEFAULT 0",      # Execution counter tracking how often this node was invoked
    "is_active": "INTEGER DEFAULT 1",            # Exclusive flag: Only active rows (1) are read; old/replaced versions are set to 0
    "year": "INTEGER",                           # System Auto-Fill: Creation or modification year
    "month": "INTEGER",                          # System Auto-Fill: Month
    "day": "INTEGER",                            # System Auto-Fill: Day
    "hour": "INTEGER",                           # System Auto-Fill: Hour
    "minute": "INTEGER",                         # System Auto-Fill: Minute
    "second": "INTEGER"                          # System Auto-Fill: Second
}

In [22]:
# [BOOTSTRAP-ABGLEICH] Verbindung sichern und Meta-Registry aktualisieren/einbinden
print("--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---")

# Schritt 1: Ordnerstruktur und exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, erweiterte Routing-Tabelle (Meta Admin Bootstrap Registry) prüfen, 
# bestehende Tabellen schützen und fehlende Spalten (inkl. Versionierung & Aktiv-Flags) ergänzen
create_routing_tree_table(db_path)

print(f"\n[ERFOLG] Die Meta-Registry ist aktiv, das globale Schema ist aktualisiert und die Datenbank ist bereit: {db_path}")

--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und init

In [23]:
# Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "meta_admin_taboo_rules"


In [24]:
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "parent_

In [25]:
NAME_ROWS = [
    {
        "node_id": "000",
        "parent_node_id": "root",
        "directive_name": "SYSTEM_BOOT_AND_GOVERNANCE",
        "mandatory_payload": "LOAD_TABOO_MEMORY; Enforce absolute system governance and strict adherence to rule chains.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "001",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "001",
        "parent_node_id": "000",
        "directive_name": "ANTI_CODE_SHRINKING",
        "mandatory_payload": "PRESERVE_CORE_LOGIC; Never delete, shrink, or remove existing baseline functions when writing code.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "002",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "002",
        "parent_node_id": "001",
        "directive_name": "ADDITIVE_MODIFICATION_ONLY",
        "mandatory_payload": "APPLY_ADDITIVE_UPDATES; Always introduce new capabilities as new versions or extensions without destroying past logic.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "003",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "003",
        "parent_node_id": "002",
        "directive_name": "PROHIBIT_SCHEMA_DROPS",
        "mandatory_payload": "NO_DROP_OR_TRUNCATE; Never execute DROP TABLE, DROP DATABASE, or TRUNCATE commands on core system tables.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "004",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "004",
        "parent_node_id": "003",
        "directive_name": "PROHIBIT_HARD_DELETES",
        "mandatory_payload": "NO_HARD_DELETE; Never use physical DELETE statements on historical or core configuration records; use versioning and is_active=0 instead.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "005",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "005",
        "parent_node_id": "004",
        "directive_name": "PROHIBIT_UNVALIDATED_SCHEMA_ALTERATION",
        "mandatory_payload": "NO_BLIND_ALTER; Never modify existing table columns or data types without verifying backward compatibility and schema integrity hashes.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "006",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "006",
        "parent_node_id": "005",
        "directive_name": "PROHIBIT_INFINITE_LOOPS",
        "mandatory_payload": "NO_RECURSIVE_LOCK; Never create circular dependency routing or self-referencing loops that bypass the FX_CHAIN recovery mechanism.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_taboo_rules",
        "target_node_id": "999",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "999",
        "parent_node_id": "006",
        "directive_name": "LOOP_TERMINATOR_AND_RETURN",
        "mandatory_payload": "EXIT_TABOO_CHAIN; Finalize memory loading and safely return execution control to the main administration registry.",
        "fallback_action": "FX_CHAIN",
        "target_table": "meta_admin_bootstrap_registry",
        "target_node_id": "001",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    }
]

In [26]:
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Injiziere Zeilen-Batch in Tabelle: 'meta_admin_taboo_rules' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] Ziel-Da

# meta_admin_repair_registry

In [27]:
# Global variables & schema definitions: TABLE 2 (Meta Admin Taboo Rules / Long-Term Memory)
TABLE_ROUTING = "meta_admin_repair_registry"

ROUTING_COLUMNS = {
    "node_id": "TEXT",                      # Stays constant (e.g., "000"), identifies the logical branch (no PK to allow multiple versions)
    "parent_node_id": "TEXT",               # Parent node for traceability and lineage
    "node_purpose": "TEXT NOT NULL",        # Detailed purpose description of this admin branch
    "routing_condition": "TEXT NOT NULL",   # Condition for the LLM: Checks admin rights and write protection (SAVE_RITE)
    "action_payload": "TEXT NOT NULL",      # Instructions and payload for accessing this administrative level
    "target_table": "TEXT",                 # Target table name (protected base table or timestamp-suffixed version)
    "target_node_id": "TEXT",               # Start ID within the target table (e.g., "root_v1")
    "version": "INTEGER DEFAULT 1",         # Explicit version number (v1, v2, v3...) for safe tracking
    "success_weight": "REAL DEFAULT 1.0",   # Reliability / success weight factor of this entry
    "execution_count": "INTEGER DEFAULT 0", # Execution counter tracking how often this node was invoked
    "is_active": "INTEGER DEFAULT 1",       # Exclusive flag: Only ONE row per node_id is allowed to be active (1 = active, 0 = inactive)
    "year": "INTEGER",                      # System Auto-Fill: Creation or modification year
    "month": "INTEGER",                     # System Auto-Fill: Month
    "day": "INTEGER",                       # System Auto-Fill: Day
    "hour": "INTEGER",                      # System Auto-Fill: Hour
    "minute": "INTEGER",                    # System Auto-Fill: Minute
    "second": "INTEGER"                     # System Auto-Fill: Second
}

In [28]:
# [BOOTSTRAP-ABGLEICH] Verbindung sichern und Meta-Registry aktualisieren/einbinden
print("--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---")

# Schritt 1: Ordnerstruktur und exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, erweiterte Routing-Tabelle (Meta Admin Bootstrap Registry) prüfen, 
# bestehende Tabellen schützen und fehlende Spalten (inkl. Versionierung & Aktiv-Flags) ergänzen
create_routing_tree_table(db_path)

print(f"\n[ERFOLG] Die Meta-Registry ist aktiv, das globale Schema ist aktualisiert und die Datenbank ist bereit: {db_path}")

--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und init

In [29]:
# Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "meta_admin_repair_registry"


In [30]:
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "parent_

In [31]:
NAME_ROWS = {
    "node_id": '000',
    "parent_node_id": 'root',
    "node_purpose": 'Primary bridge head from Repair Registry into the FX Chain sequence table.',
    "routing_condition": 'EXECUTE_PYRAMID_DESCENT; Verify agent alignment and authorize clean handoff.',
    "action_payload": 'Transfer control flow directly to the FX_CHAIN table structure without intermediate fallback loops.',
    "target_table": 'FX_CHAIN',
    "target_node_id": '000',
    "version": 1,
    "success_weight": 1.0,
    "execution_count": 0,
    "is_active": 1,
    "year": 0,
    "month": 0,
    "day": 0,
    "hour": 0,
    "minute": 0,
    "second": 0
}

In [32]:
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Injiziere Zeilen-Batch in Tabelle: 'meta_admin_repair_registry' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] Zie

# FX_CHAIN

In [33]:
# Global variables & schema definitions: TABLE 2 (Meta Admin Taboo Rules / Long-Term Memory)
TABLE_ROUTING = "FX_CHAIN"

ROUTING_COLUMNS = {
    # 1. STRUKTUR & REIHENFOLGE (Control Flow & Lineage)
    "node_id": "TEXT NOT NULL",              # Der aktuelle Schritt im Loop (z.B. "000", "001", "002"...) -> Definiert die exakte Ausführungsreihenfolge.
    "parent_node_id": "TEXT",                # Der direkte Vorgänger-Schritt, um die Kette rückwärts zu prüfen und Verzweigungen zu validieren.
    
    # 2. KOGNITIVES VERHALTEN & ENTSCHEIDUNG (Agent Reasoning & Task)
    "execution_phase": "TEXT NOT NULL",      # Was genau tut dieser Schritt? (z.B. SCAN_ERRORS, VALIDATE_SEQUENCE, HEAL_GAP).
    "agent_instruction": "TEXT NOT NULL",    # Die exakte Arbeitsanweisung (Prompt-Payload), die das LLM ausführen muss.
    "validation_rule": "TEXT NOT NULL",      # Die Bedingung, die erfüllt sein muss, bevor der Agent zum nächsten node_id übergeht.
    
    # 3. SICHERHEIT & FEHLERBEHANDLUNG (Guardrails & Fallback)
    "sequence_break_action": "TEXT DEFAULT 'ABLAUF_UNTERBROCHEN_AUFGRUND_FEHLVERHALTEN'", # Greift sofort, wenn der Agent die Reihenfolge bricht oder stolpert.
    
    # 4. SPRUNG-ZIEL (Transition Handoff)
    "target_table": "TEXT",                  # Ziel-Tabelle nach erfolgreichem Abschluss dieses Loops (z.B. Rücksprung in den Hauptbaum).
    "target_node_id": "TEXT",                # Der exakte Ziel-Knoten in der nächsten Tabelle.
    
    # 5. METRIKEN & SYSTEM-AUTO-FILL (State Tracking)
    "version": "INTEGER DEFAULT 1",          # Versionskontrolle für den Reparaturschritt.
    "success_weight": "REAL DEFAULT 1.0",    # Gewichtung der Zuverlässigkeit dieses Schritts.
    "execution_count": "INTEGER DEFAULT 0",  # Wie oft wurde dieser Schritt vom Agenten durchlaufen?
    "is_active": "INTEGER DEFAULT 1",        # Nur aktive Schritte (1) werden vom Agenten gelesen.
    "year": "INTEGER",                       # System-Auto-Fill (Zeitstempel-Validierung)
    "month": "INTEGER",                      # System-Auto-Fill
    "day": "INTEGER",                        # System-Auto-Fill
    "hour": "INTEGER",                       # System-Auto-Fill
    "minute": "INTEGER",                     # System-Auto-Fill
    "second": "INTEGER"                      # System-Auto-Fill
}

In [34]:
# [BOOTSTRAP-ABGLEICH] Verbindung sichern und Meta-Registry aktualisieren/einbinden
print("--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---")

# Schritt 1: Ordnerstruktur und exakten Pfad zur bestehenden Datenbank abrufen
db_path = initialize_find_folder()

# Schritt 2: Die neue, erweiterte Routing-Tabelle (Meta Admin Bootstrap Registry) prüfen, 
# bestehende Tabellen schützen und fehlende Spalten (inkl. Versionierung & Aktiv-Flags) ergänzen
create_routing_tree_table(db_path)

print(f"\n[ERFOLG] Die Meta-Registry ist aktiv, das globale Schema ist aktualisiert und die Datenbank ist bereit: {db_path}")

--- [START] Führe Registry-Update und Tabellen-Einbindung durch ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--- [START] Prüfe und init

In [35]:
# Globale Table Rite Definitionen 
# True verhindert das Überschreiben der Tabelle, wenn sie bereits existiert. False würde die Tabelle löschen und neu erstellen.
SAVE_RITE = True
TABLE_NAME = "FX_CHAIN"


In [36]:
analyze_table_structure(TABLE_NAME, save_rite_status=SAVE_RITE)

--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
NAME_ROWS= {
    "node_id": '',  # Format: Text / String (z.B.Textbeschreibungen)
    "parent_

In [38]:
NAME_ROWS = [
    {
        "node_id": "000",
        "parent_node_id": "root",
        "execution_phase": "INIT_LOOP_AND_IDENTIFY_BROKEN_TABLE",
        "agent_instruction": "START LOOP: Read the target table where the sequence break occurred. Establish the operational scope for table-wide node iteration.",
        "validation_rule": "target_table IS NOT NULL AND table_exists(target_table) == TRUE",
        "sequence_break_action": "ABLAUF_UNTERBROCHEN_AUFGRUND_FEHLVERHALTEN",
        "target_table": "fx_chain",
        "target_node_id": "001",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "001",
        "parent_node_id": "000",
        "execution_phase": "ITERATE_EACH_NODE_AND_CHECK_ACTIVE",
        "agent_instruction": "WHILE nodes remain unverified: Select the current node_id (starting from '000'). Check all rows matching this node_id. Evaluate 'is_active' (1 or 0) and parse the precise timestamp (year, month, day, hour, minute, second).",
        "validation_rule": "current_node_scanned == TRUE",
        "sequence_break_action": "ABLAUF_UNTERBROCHEN_AUFGRUND_FEHLVERHALTEN",
        "target_table": "fx_chain",
        "target_node_id": "002",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "002",
        "parent_node_id": "001",
        "execution_phase": "COMPARE_TIMESTAMPS_AND_RESOLVE_DUPLICATES",
        "agent_instruction": "FOR duplicates of the same node_id: Compare their timestamps. Deactivate older entries by setting 'is_active = 0'. Keep only the absolute newest entry active by setting 'is_active = 1'. If a node has no active entry at all, reactivate its latest valid version.",
        "validation_rule": "exactly_one_active_row_per_node_id == TRUE",
        "sequence_break_action": "ABLAUF_UNTERBROCHEN_AUFGRUND_FEHLVERHALTEN",
        "target_table": "fx_chain",
        "target_node_id": "003",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "003",
        "parent_node_id": "002",
        "execution_phase": "CHECK_LOOP_COMPLETION",
        "agent_instruction": "CHECK LOOP CONDITION: Are there any unverified node steps remaining in the table? If YES, loop back to node_id '001' for the next sequence index. If NO (all rows from start to end are scanned and cleanly linked), proceed to node '999'.",
        "validation_rule": "all_table_nodes_fully_validated == TRUE",
        "sequence_break_action": "ABLAUF_UNTERBROCHEN_AUFGRUND_FEHLVERHALTEN",
        "target_table": "fx_chain",
        "target_node_id": "999",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    },
    {
        "node_id": "999",
        "parent_node_id": "003",
        "execution_phase": "TERMINATE_LOOP_AND_RESTART_USER_INTENT",
        "agent_instruction": "TERMINATE REPAIR LOOP: The sequence table is now completely healed, synchronized, and free of contradictions. Exit the fallback loop, jump back to the main routing registry, and re-execute the user's complete original workflow from index '000' onward.",
        "validation_rule": "table_chain_integrity_verified AND return_to_main_registry_authorized == TRUE",
        "sequence_break_action": "ABLAUF_UNTERBROCHEN_AUFGRUND_FEHLVERHALTEN",
        "target_table": "meta_admin_bootstrap_registry",
        "target_node_id": "000",
        "version": 1,
        "success_weight": 1.0,
        "execution_count": 0,
        "is_active": 1,
        "year": 0, "month": 0, "day": 0, "hour": 0, "minute": 0, "second": 0
    }
]

In [39]:
insert_bootstrap_routing_rows(TABLE_NAME, NAME_ROWS)

--- [START] Injiziere Zeilen-Batch in Tabelle: 'FX_CHAIN' ---
--- [START] Prüfe Ordnerstruktur für den Knowledge Agent ---
--> [INFO] Start-Pfad des Notebooks: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\notebooks
--> [ERFOLG] Projekt-Root 'Offline_AI' identifiziert unter: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI
Suche nach Hauptverzeichnis: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge'...
--> [INFO] Hauptverzeichnis 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge' wurde gefunden.
Suche nach Unterordner: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql'...
--> [INFO] Unterordner 'knowledge_agent_hierarchical_routing_tree_sql' existiert bereits.
Pfad-Zusammenführung abgeschlossen. Zieldatei: 'c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\Knowledge\knowledge_agent_hierarchical_routing_tree_sql\knowledge_agent_routing_tree.db'
--- [ENDE] Ordnerstruktur erfolgreich geprüft ---
--> [INFO] Ziel-Datenbank: c:\Us